# Destocking-Timing Model (Livestock) — Starter Notebook

**The question:** during a drought, a herder has to decide *when* to sell animals. Sell too early and you lose value. Sell too late and animals die, or you sell into a market that has already crashed because everyone else sold at the same time.

**What this notebook does, step by step:**
1. Builds a **synthetic** (made-up but realistic-shaped) drought + price history, so we have something to work with today.
2. Defines the decision problem in plain terms (what the herder sees, what they can do, what counts as good).
3. Creates "logged" past decisions, like the historical data we would get in real life.
4. Trains a policy **offline** (only from logged data, never by experimenting on real herders).
5. Compares it against simple rules.
6. Exports a small `policy_grid.json` file that your `front_end` dashboard can read.

> ⚠️ **Honest warning:** every number here is synthetic. Results show the *method works*, not what is true in Borana. Section 1 marks exactly where to swap in real CHIRPS and real market-price data.

In [ ]:
import json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesRegressor

rng = np.random.default_rng(42)   # fixed seed so results repeat

## 1. Make the data (synthetic placeholder)

We create 30 years (1993–2022) of monthly data:
- **rain** with two rainy seasons (Mar–May long rains, Oct–Nov short rains) and some bad years
- **spi**: a drought score. Below −1 means drought, below −1.5 means severe
- **forage**: how much grass is available (like NDVI)
- **pressure**: how many *other* households are selling at once (the regional signal one household can't see)

**To use real data later:** replace this whole cell with CHIRPS rainfall turned into SPI, an NDVI series, and a real market-price series. Keep the same column names and everything below still works.

In [ ]:
def make_history(start_year=1993, end_year=2022, seed=42):
    r = np.random.default_rng(seed)
    months = pd.date_range(f"{start_year}-01-01", f"{end_year}-12-01", freq="MS")
    n = len(months)

    # seasonal rain pattern: wet in Mar-May and Oct-Nov
    m = months.month.values
    season = 40 + 90*np.isin(m, [3,4,5]) + 60*np.isin(m, [10,11])

    # year-to-year wetness wanders slowly; some years are dry
    anomaly = np.zeros(n)
    for i in range(1, n):
        anomaly[i] = 0.93*anomaly[i-1] + r.normal(0, 0.28)
    rain = np.clip(season * np.exp(0.5*anomaly) + r.normal(0, 8, n), 0, None)

    # SPI-like drought score: 6-month rain, standardised (negative = drier than normal)
    roll = pd.Series(rain).rolling(6, min_periods=1).sum()
    spi = ((roll - roll.groupby(m).transform("mean")) / roll.groupby(m).transform("std")).fillna(0).values

    # forage follows the drought score with a short delay
    forage = np.clip(0.55 + 0.18*pd.Series(spi).shift(1).bfill().values + r.normal(0, 0.03, n), 0.05, 1.0)

    # selling pressure builds when drought is bad, and fades slowly
    pressure = np.zeros(n)
    for i in range(1, n):
        push = max(0, -spi[i]-0.5) / 2.0
        pressure[i] = np.clip(0.8*pressure[i-1] + 0.35*push + r.normal(0, 0.01), 0, 1)

    # price: healthy animals sell high; crowded markets pay less
    base = 100 + 0.15*np.arange(n)/12*10
    price = base * (0.55 + 0.45*forage) * (1 - 0.45*pressure) * np.exp(r.normal(0, 0.02, n))

    return pd.DataFrame({"date": months, "year": months.year, "month": m,
                         "rain": rain, "spi": spi, "forage": forage,
                         "pressure": pressure, "price": price})

hist = make_history()
hist.head()

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
ax[0].plot(hist.date, hist.spi);       ax[0].axhline(-1, color="r", ls="--"); ax[0].set_ylabel("drought score (SPI)")
ax[1].plot(hist.date, hist.pressure);  ax[1].set_ylabel("regional selling pressure")
ax[2].plot(hist.date, hist.price);     ax[2].set_ylabel("price per animal")
plt.tight_layout(); plt.show()

## 2. Define the decision problem (in plain words)

Each month, one household with a herd looks at the situation and picks **one action**:

| Action | Meaning |
|---|---|
| 0 — Hold | keep all animals |
| 1 — Sell 25% | sell a quarter of the herd now |
| 2 — Sell 50% | sell half the herd now |

**What the herder sees (the "state"):** drought score, forage, price trend, regional selling pressure, how much of the herd is left, and the time of year.

**What counts as good (the "reward"):** money from sales, plus the value of animals still alive at the end of the 12 months. Animals that die are worth nothing. Healthy herds also grow a little each month, so holding is better when there is no drought.

This tension is the whole problem: **holding risks death, selling early loses value, selling when everyone else sells gets a crash price.**

In [ ]:
ACTIONS = [0.0, 0.25, 0.50]      # fraction of herd sold for actions 0, 1, 2
HORIZON = 12                      # months per decision episode
FEATURES = ["spi", "forage", "price_trend", "pressure", "herd_left", "month_sin", "month_cos", "months_left"]

def get_state(h, t, herd_left, k):
    row = h.iloc[t]
    prev = h.price.iloc[max(t-3, 0)]
    ang = 2*np.pi*row.month/12
    return np.array([row.spi, row.forage, row.price/prev, row.pressure,
                     herd_left, np.sin(ang), np.cos(ang), (HORIZON-k)/HORIZON])

def step(h, t, herd_left, action, r):
    """Advance one month. Returns reward, new herd size."""
    row = h.iloc[t]
    sold = herd_left * ACTIONS[action]
    cash = sold * row.price / 100.0                     # reward in 'herd-value' units
    kept = herd_left - sold
    death_rate = 0.003 + 0.09*(1 - row.forage)**2       # poor forage kills animals
    survivors = kept * (1 - death_rate) * (1 + 0.008)    # tiny monthly growth
    return cash, survivors

def final_value(h, t, herd_left):
    return herd_left * h.price.iloc[min(t, len(h)-1)] / 100.0

## 3. Create "logged" past decisions

In real life we can't run experiments on herders, so we learn from **records of what people already did**. Here we pretend a crowd of households acted semi-randomly, and we record every (situation → action → result).

We only log episodes starting in **1993–2014**. The years **2015–2022 stay hidden** as a test set, so we can check the policy on droughts it never saw.

In [ ]:
def run_episode(h, start, policy, r):
    herd, k = 1.0, 0
    log, total = [], 0.0
    for k in range(HORIZON):
        t = start + k
        s = get_state(h, t, herd, k)
        a = policy(s, r)
        cash, new_herd = step(h, t, herd, a, r)
        done = (k == HORIZON-1)
        reward = cash + (final_value(h, t+1, new_herd) if done else 0.0)
        s2 = get_state(h, t+1, new_herd, k+1)
        log.append((s, a, reward, s2, done))
        total += reward
        herd = new_herd
    return log, total

def behaviour_policy(s, r):
    # 'what people did': mostly hold, sometimes sell, a little more when drought is bad
    p_sell = 0.10 + 0.25*max(0, -s[0]-0.5)
    if r.random() < p_sell:
        return int(r.choice([1, 2]))
    return 0

train_starts = np.where((hist.year <= 2014) & (hist.index < len(hist)-HORIZON-2))[0]
test_starts  = np.where((hist.year >= 2015) & (hist.index < len(hist)-HORIZON-2))[0]

transitions = []
for _ in range(4000):
    st = int(rng.choice(train_starts))
    log, _ = run_episode(hist, st, behaviour_policy, rng)
    transitions += log

S  = np.array([x[0] for x in transitions]); A  = np.array([x[1] for x in transitions])
R  = np.array([x[2] for x in transitions]); S2 = np.array([x[3] for x in transitions])
D  = np.array([x[4] for x in transitions], dtype=float)
print("logged transitions:", len(S))

## 4. Train the policy offline (Fitted Q-Iteration)

Idea in simple terms: learn a score **Q(situation, action)** = "how much total value do I expect if I take this action now and act well afterwards?" Then the policy just picks the action with the highest score.

We do this in repeated rounds: each round uses the previous round's scores to learn better ones.

> This is the simple stepping-stone version. The plan document uses **BCQ/CQL**, which add a safety brake so the model doesn't trust actions that were rarely seen in the logs. We can upgrade to that next. With sparse real data that brake matters a lot.

In [ ]:
def make_X(S, A):
    return np.hstack([S, np.eye(3)[A]])

gamma = 0.99
Q = None
for it in range(8):
    if Q is None:
        y = R
    else:
        q_next = np.column_stack([Q.predict(make_X(S2, np.full(len(S2), a))) for a in range(3)]).max(axis=1)
        y = R + gamma * (1 - D) * q_next
    Q = ExtraTreesRegressor(n_estimators=60, min_samples_leaf=10, n_jobs=-1, random_state=0)
    Q.fit(make_X(S, A), y)
    print(f"round {it+1} done")

def q_values(s):
    return np.array([Q.predict(make_X(s[None, :], np.array([a])))[0] for a in range(3)])

def learned_policy(s, r=None):
    return int(np.argmax(q_values(s)))

## 5. Compare against simple rules on the hidden years (2015–2022)

If the learned policy can't beat simple rules, it isn't worth building. Higher number = more herd value at the end of the year.

In [ ]:
def always_hold(s, r=None):  return 0
def sell_half_now(s, r=None): return 2 if s[7] > 0.9 else 0
def drought_rule(s, r=None):  # sell 25% whenever drought score is below -1
    return 1 if s[0] < -1.0 else 0

def evaluate(policy, starts, n=400):
    r = np.random.default_rng(7)
    picks = r.choice(starts, size=n)
    return np.array([run_episode(hist, int(st), policy, r)[1] for st in picks])

results = {
    "Always hold":          evaluate(always_hold, test_starts),
    "Sell half at start":   evaluate(sell_half_now, test_starts),
    "Sell 25% if SPI < -1": evaluate(drought_rule, test_starts),
    "Learned (offline RL)": evaluate(learned_policy, test_starts, n=150),
}
summary = pd.DataFrame({k: {"mean value": v.mean(), "worst 10%": np.percentile(v, 10)} for k, v in results.items()}).T
summary.round(3)

In [ ]:
# Same comparison, but ONLY for the worst drought windows (SPI very low) — where timing matters most
dry = [i for i in test_starts if hist.spi.iloc[i] < -1.0]
print("drought-start windows in test years:", len(dry))
if len(dry) > 20:
    dry_res = {"Always hold": evaluate(always_hold, dry), "Sell 25% if SPI < -1": evaluate(drought_rule, dry),
               "Learned (offline RL)": evaluate(learned_policy, dry, n=100)}
    print(pd.DataFrame({k: {"mean value": v.mean()} for k, v in dry_res.items()}).T.round(3))

## 6. Export for the front end

The `front_end` dashboard is plain HTML + JavaScript, so it can't run Python. Easy bridge: we precompute the recommendation for a **grid** of situations (drought score × regional pressure × forage) and save it as JSON. The dashboard just looks up the closest cell.

Each cell holds the recommended action and a **confidence** (how much better the best action scored than the second best).

In [ ]:
spi_grid   = [-2.5, -2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0]
press_grid = [0.0, 0.2, 0.4, 0.6, 0.8]
forage_grid = [0.2, 0.4, 0.6, 0.8]
LABELS = ["hold", "sell_25", "sell_50"]

cells = []
for spi in spi_grid:
    for pr in press_grid:
        for fo in forage_grid:
            s = np.array([spi, fo, 1.0, pr, 1.0, 0.0, 1.0, 0.5])   # typical herd, mid-year, flat prices
            q = q_values(s); order = np.argsort(q)[::-1]
            gap = (q[order[0]] - q[order[1]]) / (abs(q[order[0]]) + 1e-9)
            cells.append({"spi": spi, "pressure": pr, "forage": fo,
                          "action": LABELS[order[0]],
                          "confidence": float(np.clip(0.5 + 5*gap, 0.5, 0.99))})

with open("policy_grid.json", "w") as f:
    json.dump({"note": "SYNTHETIC-trained demo policy. Replace with real-data version.",
               "actions": LABELS, "cells": cells}, f, indent=1)
print(len(cells), "cells saved to policy_grid.json")
pd.DataFrame(cells).groupby("action").size()

In [ ]:
# In Colab: download the file so you can drop it into front_end/data/
try:
    from google.colab import files
    files.download("policy_grid.json")
except ImportError:
    print("Not in Colab — file is saved next to the notebook.")

## Next steps
1. Swap Section 1 for **real CHIRPS-based SPI**, an NDVI series, and any **real market-price series** you can find. This is the hardest data part.
2. Upgrade Section 4 from Fitted-Q to **CQL/BCQ** (the safety brake for rare actions).
3. Add real **off-policy evaluation** instead of testing only in our own simulator.
4. Wire `policy_grid.json` into the dashboard's *Adaptive Policy* page.